In [29]:
from intake_esgf import ESGFCatalog
import numpy as np
import xarray as xr
import xesmf as xe

def anomaly_calculation(model, clim):
    ano_list = []
    for i in range(1,13):
        ano_list.append(model.sel(time=model.time[model.time.dt.month.isin(i)]) - clim.sel(month=i))
    return xr.concat(ano_list, dim='time').sortby('time')

def regrid(in_grid,out_grid, var): 
    reg_bil = xe.Regridder(in_grid,out_grid, "bilinear")
    var_bil = reg_bil(in_grid[var])
    return var_bil

def clim(ds):
    return ds.groupby('time.month').mean('time')


def global_mean(data, lat='lat', lon='lon'):
    weights_lat_model = np.cos(np.deg2rad(data[lat]))
    try:
        mean = data.weighted(weights_lat_model).mean([lat, lon])
    except ValueError:
        mean = data.weighted(weights_lat_model).mean(lat)
    return mean


def shift_lon(ds, lon_name = 'lon'):
    # Adjust lon values to make sure they are within (-180, 180)
    ds['_longitude_adjusted'] = xr.where(
        ds[lon_name] > 180,
        ds[lon_name] - 360,
        ds[lon_name])

    # reassign the new coords to as the main lon coords
    # and sort DataArray using new coordinate values
    ds = (
        ds
        .swap_dims({lon_name: '_longitude_adjusted'})
        .sel(**{'_longitude_adjusted': sorted(ds._longitude_adjusted)})
        .drop_vars(lon_name))

    ds = ds.rename({'_longitude_adjusted': lon_name})
    return ds

def calculate_nino34_index(ds):
    # ds needs to be tos-anomalies
    # 170° - 120°W -> 190 - 240°E
    nino = ds.sel(lon=slice(190,240), lat=slice(-5,5))
    nino_gm = global_mean(nino).compute()
    return nino_gm

def calculate_nao(ds):
    ds_south = ds.sel(lat=slice(20,55), lon=slice(-90,60))
    ds_south = global_mean(ds_south).compute()
    ds_north = ds.sel(lat=slice(55,90), lon=slice(-90,60)).mean(['lon','lat'])
    ds_north = global_mean(ds_north).compute()
    nao = (ds_south - ds_north)
    nao = nao.zg.to_dataset(name='zg')
    return nao


def calculate_djf_mean(ds):
    ds_seasons = ds.resample(time='QE-FEB').mean()
    ds_djf = ds_seasons.sel(time=(ds_seasons['time.season'] == 'DJF'))
    return ds_djf


In [30]:

outdir = "/sto4/data/Intermediate/VolMIP/"
models = ['CanESM5', 'MIROC-ES2L', 'MPI-ESM1-2-LR', 'GISS-E2-1-G', 'UKESM1-0-LL']
member_id = ['r1i1p2f1', 'r1i1p1f2', 'r1i1p1f1', 'r1i1p1f2', 'r1i1p1f2']

In [31]:
model=models[0]
id = member_id[0]

In [ ]:
cat=ESGFCatalog()

In [ ]:
for model, id in zip(models, member_id):

    cat=ESGFCatalog()
    metagrid_search = cat.search(activity_id='CMIP', experiment_id='piControl', source_id=model,
           variable_id=['ta', 'ua'], latest=True, table_id=['Amon'], member_id=id, grid_label='gn')
    print(metagrid_search)
    ds = metagrid_search.to_path_dict(outdir + 'piControl/' + model + '/piControl_' + model + '_NAO_djf_'+ start + '-' + end + '.nc')

In [ ]:
ds

In [ ]:
time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
ds_ta = xr.open_mfdataset(ds['ta'], decode_times=time_coder)

In [ ]:
ta_zonal = ds_ta.mean('lon')
nino.to_netcdf(outdir + 'piControl/' + model + '/ta.nc')


In [ ]:
time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
ds_tos = xr.open_mfdataset(ds['Omon.tos'], decode_times=time_coder)

# regrid to regular lat lon grid
ds_tos_reg = regrid(ds_tos, ds_zg, 'tos')

# sst climatology
tos_clim = clim(ds_tos_reg)
tos_clim.to_netcdf(outdir + 'piControl/' + model + '/tos_clim_full.nc')

# calculate nino index
nino = calculate_nino34_index(ds_tos_reg)
nino = nino.to_dataset(name='tos')
nino_djf = calculate_djf_mean(nino)

# build anomalies & calculate rsst nino index
ds_ano = anomaly_calculation(ds_tos_reg, tos_clim)
rsstnino = calculate_nino34_index(ds_ano)
rsstnino = rsstnino.to_dataset(name='tos')
rsstnino_djf = calculate_djf_mean(rsstnino)

nino.to_netcdf(outdir + 'piControl/' + model + '/nino34_full.nc')
nino_djf.to_netcdf(outdir + 'piControl/' + model + '/nino34_djf.nc')

rsstnino.to_netcdf(outdir + 'piControl/' + model + '/rsstnino34_full.nc')
rsstnino_djf.to_netcdf(outdir + 'piControl/' + model + '/rsstnino34_djf.nc')

In [ ]:
str(ds_zg.time[0].values)

In [ ]:
# shift lon to °W-°E so there is no problem at 0° 
ds_prep = shift_lon(ds_zg.sel(plev=50000, lat=slice(0,100)))

# calculate nao index
nao = calculate_nao(ds_prep)
nao_djf = calculate_djf_mean(nao)

nao.to_netcdf(outdir + 'piControl/' + model + '/nao_full.nc')
nao_djf.to_netcdf(outdir + 'piControl/' + model + '/nao_djf.nc')

In [ ]:
from intake_esgf import ESGFCatalog
cat=ESGFCatalog()

metagrid_search=cat.search(activity_id='VolMIP', experiment_id='volc-pinatubo-full', source_id=models[4],
                           variable_id=['ta', 'ua'], table_id='Eday', latest=True)
metagrid_search

In [ ]:
ds = metagrid_search.to_path_dict()

In [ ]:
ds

In [ ]:
xr.open_dataset('/sto4/data/Intermediate/VolMIP/pinatubo-full/GISS-E2-1-G/ta_Amon_GISS-E2-1-G_volc-pinatubo-full_r1i1p1f1_gn_800001-800212.nc')

In [ ]:
import xarray as xr

#ds.values()
ds_ta = xr.open_mfdataset(np.concatenate(list(ds.values())), concat_dim='ens', combine='nested')

In [ ]:
ds_ta

In [ ]:
nao = nao.zg.to_dataset(name='zg')
nao.to_netcdf(outdir + 'piControl/' + model + '/nao_full.nc')

In [ ]:
outdir = "/sto4/data/Intermediate/VolMIP/"
models = ['CanESM5', 'MIROC-ES2L', 'MPI-ESM1-2-LR', 'GISS-E2-1-G', 'UKESM1-0-LL']
# according to member id in VOLMIP!!
member_id = ['r1i1p2f1', 'r1i1p1f2', 'r1i1p1f1', 'r1i1p1f1', 'r1i1p1f2']

for model, id in zip(models, member_id):
    print(model, id)
    # open ESGF catalog
    cat=ESGFCatalog()
    metagrid_search = cat.search(activity_id='CMIP', experiment_id='piControl', source_id=model,
            variable_id=['tos', 'zg'], latest=True, table_id=['Omon','Amon'], member_id=id)
    print(metagrid_search)

In [ ]:
xr.open_dataset(outdir + 'piControl/' + model + '/nino34_djf.nc')